In [10]:
import sys
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import geopandas as gpd
import h3
import pygeohash as pgh
from shapely import wkb, wkt


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("GeospatialSilverETL")


In [11]:
RAW_ZONES_GEO_PATH = "hdfs:///tlc/raw/zones/taxi_zones.parquet"
SILVER_CLEAN_TRIPS_PATH = "hdfs:///tlc/silver/trip_clean_data"
SILVER_OUTPUT_GEO_PATH = "hdfs:///tlc/silver/staging_rides_geo"

DEV_MODE = False 
DEV_SAMPLE_SIZE = 100_000
SAMPLE_SEED = 42

H3_RESOLUTION = 8         
GEOHASH_PRECISION = 6   

In [12]:


def get_spark_session() -> SparkSession:
    """Initialize SparkSession with optimal configurations for memory and AQE."""
    return (
        SparkSession.builder
        .appName("TLC-HVFHV-Geospatial-Silver")
        .master("local[4]")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.driver.memory", "6g")
        .config("spark.executor.memory", "6g")
        .getOrCreate()
    )


In [13]:

def precompute_spatial_zones(spark: SparkSession, zones_taxi_path: str):
    """
    Reads the taxi zones parquet directly via Spark from HDFS (avoiding fsspec error),
    converts to GeoDataFrame, reprojects CRS from EPSG:2263 to EPSG:4326 (WGS84),
    and precomputes H3 & Geohash for all ~263 zones.
    """
    logger.info(f"[Step 1] Reading zone shapefile via Spark from: {zones_taxi_path}")
    
    spark_zones = spark.read.parquet(zones_taxi_path)
    pdf = spark_zones.toPandas()

    def parse_geometry(geom):
        if geom is None:
            return None
        if isinstance(geom, (bytes, bytearray)):
            return wkb.loads(bytes(geom))
        elif isinstance(geom, str):
            return wkt.loads(geom)
        return geom

    pdf["geometry"] = pdf["geometry"].apply(parse_geometry)
    gdf = gpd.GeoDataFrame(pdf, geometry="geometry")

    if gdf.crs is None:
        gdf.set_crs(epsg=2263, inplace=True)

    if gdf.crs.to_epsg() != 4326:
        logger.info(f"[Step 1] Reprojecting CRS from {gdf.crs.to_string()} to EPSG:4326 (WGS84)...")
        gdf = gdf.to_crs(epsg=4326)

    gdf["lat"] = gdf.geometry.centroid.y
    gdf["lon"] = gdf.geometry.centroid.x

    def calc_h3(row):
        try:
            if hasattr(h3, "latlng_to_cell"):
                return h3.latlng_to_cell(row["lat"], row["lon"], H3_RESOLUTION)
            return h3.geo_to_h3(row["lat"], row["lon"], H3_RESOLUTION)
        except Exception:
            return None

    def calc_geohash(row):
        try:
            return pgh.encode(row["lat"], row["lon"], precision=GEOHASH_PRECISION)
        except Exception:
            return None

    gdf["h3_res8"] = gdf.apply(calc_h3, axis=1)
    gdf["geohash_p6"] = gdf.apply(calc_geohash, axis=1)

    loc_col = "LocationID" if "LocationID" in gdf.columns else "location_id"
    pdf_out = gdf[[loc_col, "lat", "lon", "geohash_p6", "h3_res8"]].rename(
        columns={loc_col: "zone_id"}
    )

    schema = T.StructType([
        T.StructField("zone_id", T.IntegerType(), True),
        T.StructField("centroid_lat", T.DoubleType(), True),
        T.StructField("centroid_lon", T.DoubleType(), True),
        T.StructField("geohash", T.StringType(), True),
        T.StructField("h3_index", T.StringType(), True),
    ])

    spatial_zones_df = spark.createDataFrame(pdf_out, schema=schema)
    logger.info(f"[Step 1] Done! Successfully precomputed spatial keys for {spatial_zones_df.count()} zones.")
    return spatial_zones_df


In [14]:

def enrich_trips_with_geospatial(trips_df, spatial_zones_df):
    """
    Enriches trips with geospatial coordinates and keys using Broadcast Join,
    then computes advanced features: speed, duration, and anomaly flags.
    """
    logger.info("[Step 2] Executing Broadcast Joins for Pickup & Dropoff...")

    pu_ref = spatial_zones_df.select(
        F.col("zone_id").alias("pu_location_id"),
        F.col("centroid_lat").alias("pu_lat"),
        F.col("centroid_lon").alias("pu_lon"),
        F.col("geohash").alias("start_geo_hash"),
        F.col("h3_index").alias("pu_h3_res8"),
    )

    do_ref = spatial_zones_df.select(
        F.col("zone_id").alias("do_location_id"),
        F.col("centroid_lat").alias("do_lat"),
        F.col("centroid_lon").alias("do_lon"),
        F.col("geohash").alias("end_geo_hash"),
        F.col("h3_index").alias("do_h3_res8"),
    )

    df = (
        trips_df
        .join(F.broadcast(pu_ref), on="pu_location_id", how="left")
        .join(F.broadcast(do_ref), on="do_location_id", how="left")
    )

    if "trip_id" not in df.columns:
        df = df.withColumn(
            "trip_id",
            F.concat_ws(
                "_",
                F.col("license_num"),
                F.col("pu_location_id"),
                F.col("do_location_id"),
                F.date_format(F.col("pickup_datetime"), "yyyyMMddHHmmss"),
                F.monotonically_increasing_id()
            )
        )

    df = df.withColumn(
        "avg_speed_mph",
        F.when(
            (F.col("trip_duration_sec") > 60) & (F.col("trip_miles") > 0.1),
            F.round(F.col("trip_miles") / (F.col("trip_duration_sec") / 3600.0), 2)
        ).otherwise(F.lit(0.0))
    )

    df = df.withColumn(
        "is_speed_anomaly",
        (F.col("avg_speed_mph") > 120.0) | (F.col("avg_speed_mph") < 0.0)
    )

    return df


In [15]:
def write_silver_output(df):
    """Writes partitioned Parquet dataset to Silver layer."""
    logger.info(f"[Step 3] Writing partitioned dataset to: {SILVER_OUTPUT_GEO_PATH}")
    (
        df
        .repartition(4, "year", "month", "day")
        .write
        .mode("overwrite")
        .partitionBy("year", "month", "day")
        .parquet(SILVER_OUTPUT_GEO_PATH)
    )
    logger.info("[Step 3] Successfully written Staging_Rides_Geo!")


In [16]:

def run_spatial_analytics_summary(spark: SparkSession, df):
    """Executes demonstrative Spark SQL aggregations for Gold readiness."""
    df.createOrReplaceTempView("staging_rides_geo")

    print("\n" + "=" * 80)
    print("--- SPATIAL DEMO: Top 10 Pickup H3 Clusters by Volume & Avg Speed ---")
    print("=" * 80)
    spark.sql("""
        SELECT pu_h3_res8,
               start_geo_hash,
               pu_borough,
               COUNT(*)                     AS trip_count,
               ROUND(AVG(total_fare), 2)    AS avg_fare,
               ROUND(AVG(avg_speed_mph), 1) AS avg_speed_mph
        FROM staging_rides_geo
        WHERE pu_h3_res8 IS NOT NULL
        GROUP BY pu_h3_res8, start_geo_hash, pu_borough
        ORDER BY trip_count DESC
        LIMIT 10
    """).show(truncate=False)

In [17]:

def main():
    spark = get_spark_session()
    spark.sparkContext.setLogLevel("WARN")

    logger.info("=" * 80)
    logger.info("STARTING GEOSPATIAL PROCESSING & ADVANCED SPARK (SILVER LAYER)")
    logger.info("=" * 80)

    logger.info(f"Reading cleaned trips from: {SILVER_CLEAN_TRIPS_PATH}")
    trips_df = spark.read.parquet(SILVER_CLEAN_TRIPS_PATH)

    if DEV_MODE:
        logger.info(f"DEV_MODE enabled: sampling {DEV_SAMPLE_SIZE:,} records.")
        trips_df = trips_df.sample(withReplacement=False, fraction=0.05, seed=SAMPLE_SEED).limit(DEV_SAMPLE_SIZE)

    spatial_zones_df = precompute_spatial_zones(spark, RAW_ZONES_GEO_PATH)

    enriched_df = enrich_with_geospatial(trips_df, spatial_zones_df)
    enriched_df = enriched_df.cache()

    logger.info(f"Final enriched row count: {enriched_df.count():,}")

    run_spatial_analytics_summary(spark, enriched_df)

    write_silver_output(enriched_df)

    enriched_df.unpersist()
    spark.stop()
    logger.info("Job completed successfully!")


if __name__ == "__main__":
    main()


2026-09-03 06:06:09,111 [INFO] ================================================================================
2026-09-03 06:06:09,116 [INFO] STARTING GEOSPATIAL PROCESSING & ADVANCED SPARK (SILVER LAYER)
2026-09-03 06:06:09,117 [INFO] ================================================================================
2026-09-03 06:06:09,118 [INFO] Reading cleaned trips from: hdfs:///tlc/silver/trip_clean_data
2026-09-03 06:06:10,008 [INFO] [Step 1] Reading zone shapefile via Spark from: hdfs:///tlc/raw/zones/taxi_zones.parquet
2026-09-03 06:06:15,217 [INFO] [Step 1] Reprojecting CRS from EPSG:2263 to EPSG:4326 (WGS84)...
/tmp/ipykernel_8138/3581278443.py:39: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y
/tmp/ipykernel_8138/3581278443.py:40: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' ar


[Step 2] Executing Broadcast Joins for Pickup & Dropoff...


2026-09-03 06:06:40,112 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-09-03 06:07:47,360 [INFO] Final enriched row count: 499,999                



--- SPATIAL DEMO: Top 10 Pickup H3 Clusters by Volume & Avg Speed ---


2026-09-03 06:07:53,252 [INFO] [Step 3] Writing partitioned dataset to: hdfs:///tlc/silver/staging_rides_geo


+---------------+--------------+----------+----------+--------+-------------+
|pu_h3_res8     |start_geo_hash|pu_borough|trip_count|avg_fare|avg_speed_mph|
+---------------+--------------+----------+----------+--------+-------------+
|882a100d2dfffff|dr5ru6        |Manhattan |11514     |40.79   |12.8         |
|882a100d65fffff|dr5ruk        |Manhattan |11358     |42.19   |13.3         |
|882a103b03fffff|dr5x0z        |Queens    |9740      |81.9    |27.0         |
|882a100f57fffff|dr5ryy        |Queens    |8676      |68.05   |23.0         |
|882a100d35fffff|dr5rsq        |Manhattan |6970      |30.38   |12.0         |
|882a100d95fffff|dr5rmk        |Brooklyn  |6674      |23.38   |11.3         |
|882a1072c1fffff|dr5rsj        |Manhattan |6515      |33.77   |11.2         |
|882a100d67fffff|dr5rue        |Manhattan |6106      |42.24   |11.9         |
|882a1072c7fffff|dr5reu        |Manhattan |5789      |36.2    |12.7         |
|882a107253fffff|dr5rgf        |Manhattan |5598      |37.35   |1

2026-09-03 06:08:32,771 [INFO] [Step 3] Successfully written Staging_Rides_Geo! 
2026-09-03 06:08:33,994 [INFO] Job completed successfully!
